# Rank-1 Constellations on Dual-Polarized Ray-Traced Channels

This notebook studies constellation design for **dual-polarized (2x2) channels
obtained from Sionna RT ray tracing**. Identifying the dual-pol symbol pair
x in C^2 with the real 2x2 matrix X = [[Re x1, Im x1], [Re x2, Im x2]], a
standard QPSK^2 grid places points on the vertices of a 4-cube, while a
**rank-1 (null-cone) constellation** places all points on rank-1 matrices
(det X = 0).

**Empirical findings reproduced below** (single RTX 3060 Ti, sionna-rt 2.2.0):

1. The rank-1 constellation **outperforms QPSK^2 under ML detection at high
   SNR** on ray-traced dual-pol ensembles (5/5 random seeds at 16-20 dB on
   Munich; on the Florence scene it wins at every SNR), despite a *smaller*
   minimum distance (0.765 vs 1.0).
2. A diagnostic shows a ~2x larger worst-channel effective distance for this
   rank-1 design, despite smaller average separation. This is descriptive,
   not evidence that rank-1 geometry causes the SER advantage. In a separate
   pre-registered matched-control validation, an unconstrained 64-point
   optimizer without a determinant term reached similar determinant mass and
   matched the hybrid on blind scene and stochastic XPD evaluations ([follow-up validation](https://github.com/NVlabs/sionna-rt/pull/82#issuecomment-6491527495)). The
   geometric mechanism is therefore not supported; read these results as an
   empirical engineering comparison for the tested designs and distributions.

Scope: narrowband (paths summed), perfect CSI, collocated cross-pol isotropic
ports, ML detection.

In [1]:
# Import or install Sionna RT
try:
    import sionna.rt
except ImportError as e:
    if getattr(e, "name", None) not in ("sionna", "sionna.rt"):
        raise
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sionna-rt"])
    import sionna.rt

import mitsuba as mi
mi.set_variant("cuda_ad_mono_polarized")  # required by sionna-rt 2.x

import numpy as np
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver
import sionna.rt

print("Sionna RT ready, variant:", mi.variant())

jitc_llvm_init(): LLVM API initialization failed ..


Sionna RT ready, variant: cuda_ad_mono_polarized


## 1. Dual-polarized channel ensemble from the Munich scene

In [2]:
scene = load_scene(sionna.rt.scene.munich)
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.add(Transmitter("tx", position=[8.5, 21.0, 27.0]))

n = 0
for y in np.linspace(15.0, 60.0, 8):
    for x in (25.0, 35.0):
        scene.add(Receiver(f"rx{n}", position=[float(x), float(y), 1.5]))
        n += 1

# Narrowband 2x2 polarimetric channel per receiver (paths summed)
a, _ = PathSolver()(scene, max_depth=5).cir(out_type="numpy")
H = np.asarray(a).reshape(n, 2, 2, -1).sum(-1)
alive = np.linalg.norm(H, axis=(1, 2)) > 1e-12
H = H[alive] / np.linalg.norm(H[alive], axis=(1, 2), keepdims=True)
print(f"{H.shape[0]}/{n} receivers alive")
print("Example channel (normalized):")
print(H[0] / np.linalg.norm(H[0]))

16/16 receivers alive
Example channel (normalized):
[[ 0.92982936+0.33584663j  0.01006189+0.00676795j]
 [-0.06026739-0.00641686j -0.13674566-0.01022593j]]


## 2. Constellations: QPSK^2 vs rank-1 (null cone)

Both constellations have 16 points and average energy 1. The rank-1 points
are X = v w^T with unit v on an 8-point phase grid and w in {45, -45} degrees,
so every point satisfies det(X) = 0.

In [3]:
def qpsk2():
    g = np.array([1+1j, 1-1j, -1+1j, -1-1j]) / np.sqrt(2)
    pts = np.array([[x1, x2] for x1 in g for x2 in g])
    return pts / np.sqrt((np.abs(pts)**2).sum(axis=1).mean())

def rank1():
    pts = []
    for m in range(8):
        v = np.array([np.cos(2*np.pi*m/8), np.sin(2*np.pi*m/8)])
        for w in (np.array([1.0, 1.0]), np.array([1.0, -1.0])):
            X = np.outer(v, w)
            pts.append([X[0,0]+1j*X[0,1], X[1,0]+1j*X[1,1]])
    pts = np.array(pts)
    return pts / np.sqrt((np.abs(pts)**2).sum(axis=1).mean())

CONSTS = {"qpsk2": qpsk2(), "rank1": rank1()}
for name, pts in CONSTS.items():
    d = np.sqrt((np.abs(pts[:, None] - pts[None, :])**2).sum(axis=2))
    d[d == 0] = np.inf
    print(f"{name}: d_min = {d.min():.4f}")

qpsk2: d_min = 1.0000
rank1: d_min = 0.7654


## 3. Symbol error rate under ML detection (5 seeds)

In [4]:
SEEDS, SNR_DB, NSYM = [0, 1, 2, 3, 4], [0, 4, 8, 12, 16, 20], 4000

def ser_for_seed(pts, rng):
    M = len(pts)
    out = np.zeros(len(SNR_DB))
    Y = np.array([(h @ pts.T).T for h in H])
    for si, snr in enumerate(SNR_DB):
        sigma = np.sqrt(1.0 / (2 * 10 ** (snr / 10)))
        errs = tot = 0
        for hi in range(H.shape[0]):
            nn = sigma * (rng.standard_normal((NSYM, 2)) + 1j*rng.standard_normal((NSYM, 2)))
            tx = rng.integers(0, M, NSYM)
            y = Y[hi][tx] + nn
            r = y[:, None, :] - Y[hi][None, :, :]
            cost = (np.abs(r) ** 2).sum(axis=2)
            errs += int((cost.argmin(axis=1) != tx).sum())
            tot += NSYM
        out[si] = errs / tot
    return out

res = {name: np.array([ser_for_seed(pts, np.random.default_rng(s)) for s in SEEDS])
       for name, pts in CONSTS.items()}

print(f"{'SNR':>5} {'qpsk2':>12} {'rank1':>12}  winner")
for i, snr in enumerate(SNR_DB):
    q, c = res["qpsk2"][:, i], res["rank1"][:, i]
    wins = int((c < q).sum())
    w = "rank1" if c.mean() < q.mean() else "qpsk2"
    print(f"{snr:>4} {q.mean():>12.5f} {c.mean():>12.5f}  {w} ({wins}/5 seeds)")

  SNR        qpsk2        rank1  winner
   0      0.79273      0.78648  rank1 (5/5 seeds)
   4      0.66729      0.66085  rank1 (5/5 seeds)
   8      0.46482      0.45160  rank1 (5/5 seeds)
  12      0.24443      0.21537  rank1 (5/5 seeds)
  16      0.12006      0.07405  rank1 (5/5 seeds)
  20      0.06803      0.02413  rank1 (5/5 seeds)


## 4. Diagnostic: average separation and channel-weighted distances

Compare the *mean* and the *worst-channel* smallest effective squared distance
||H (c_i - c_j)||^2. These summaries describe the tested constellation and
channel ensemble; they do not establish a causal rank-1 mechanism. A later
pre-registered unconstrained-optimizer control matched the hybrid on blind
evaluations without a determinant constraint, so do not interpret the
distance diagnostic as proof that rank-1 differences resist channel collapse.

In [5]:
def eff_min_d2(h, pts):
    D = pts[:, None, :] - pts[None, :, :]
    HD = (h @ D.reshape(-1, 2).T).T.reshape(*D.shape)
    d2 = (np.abs(HD) ** 2).sum(axis=2)
    iu = np.triu_indices(len(pts), 1)
    return np.sort(d2[iu])[0]

for name, pts in CONSTS.items():
    per_ch = np.array([eff_min_d2(h, pts) for h in H])
    print(f"{name:6s} mean d2_min = {per_ch.mean():.4f} | worst-channel d2_min = {per_ch.min():.4f}")

print()
print("Rank-1 wins at high SNR because its WORST-channel distance is ~2x larger,")
print("even though its AVERAGE distance is smaller (and d_min is smaller too).")

qpsk2  mean d2_min = 0.2320 | worst-channel d2_min = 0.0070
rank1  mean d2_min = 0.1696 | worst-channel d2_min = 0.0139

Rank-1 wins at high SNR because its WORST-channel distance is ~2x larger,
even though its AVERAGE distance is smaller (and d_min is smaller too).


## 5. Conclusions

- Under the fixed protocol shown here, the tested rank-1 constellation is a
  competitive design point and has lower SER than QPSK^2 in the reported
  high-SNR conditions.
- These comparisons do not identify rank-1/determinant structure as the
  cause: a pre-registered unconstrained 64-point control without a determinant
  term matched the hybrid on blind evaluations. The supported conclusion is
  an empirical shaping result for the tested designs, not a geometric law.
- Useful follow-ups include equal-budget optimization controls, broader
  channel distributions, frequency-selective (OFDM) evaluation, and
  non-collocated arrays.